In [1]:
import sys, tensorflow as tf, torch
print("Python:", sys.version)
print("TF:", tf.__version__, tf.config.list_physical_devices())
print("Torch:", torch.__version__, "MPS available:", torch.backends.mps.is_available())


Python: 3.11.13 (main, Jun  3 2025, 18:38:25) [Clang 17.0.0 (clang-1700.0.13.3)]
TF: 2.17.0 [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Torch: 2.8.0 MPS available: True


In [3]:
#Tensrflow device check + tiny benchmark
import tensorflow as tf, time, os
print("TF:", tf.__version__)
print("Visible GPUs:", tf.config.list_physical_devices('GPU'))

# (Optional) show where ops run
tf.debugging.set_log_device_placement(True)

# Warmup + simple matmul on CPU vs GPU
def bench_tf(device, n=4096, iters=5):
    with tf.device(device):
        a = tf.random.normal([n, n])
        b = tf.random.normal([n, n])
        # warmup
        _ = tf.linalg.matmul(a, b)
        tf.experimental.numpy.zeros(1)  # small op to keep graph happy
        t0 = time.time()
        for _ in range(iters):
            _ = tf.linalg.matmul(a, b)
        tf.experimental.numpy.zeros(1)
    return time.time() - t0

cpu_t = bench_tf("/CPU:0", iters=3)
try:
    gpu_t = bench_tf("/GPU:0", iters=3)
except Exception as e:
    gpu_t = None
    print("GPU run failed:", e)

print(f"TF CPU time: {cpu_t:.3f}s")
print(f"TF GPU time: {gpu_t:.3f}s" if gpu_t is not None else "TF GPU time: n/a")

# Quick confirmation: build a tiny model and force GPU
with tf.device("/GPU:0"):
    m = tf.keras.Sequential([tf.keras.layers.Dense(1024, activation='relu'),
                             tf.keras.layers.Dense(10)])
    x = tf.random.normal([8192, 512])
    _ = m(x)  # forward pass on GPU
print("TF forward pass done.")


TF: 2.17.0
Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Executing op RandomStandardNormal in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op Mul in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op AddV2 in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op RandomStandardNormal in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op Mul in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op AddV2 in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op MatMul in device /job:localhost/replica:0/task:0/device:CPU:0


2025-09-29 11:08:34.198005: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2025-09-29 11:08:34.198083: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-09-29 11:08:34.198097: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2025-09-29 11:08:34.198135: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-09-29 11:08:34.198160: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2025-09-29 11:08:34.216364: I tensorflow/core/common_runtime/placer.cc:162] shape: (_DeviceArg): /job:localhost/replica:0/task:0/device:CPU:0
2025-09-29 11:08:34.216412: I tensorflow/cor

Executing op Fill in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op MatMul in device /job:localhost/replica:0/task:0/device:CPU:0


2025-09-29 11:08:34.507524: I tensorflow/core/common_runtime/placer.cc:162] dims: (_DeviceArg): /job:localhost/replica:0/task:0/device:CPU:0
2025-09-29 11:08:34.507535: I tensorflow/core/common_runtime/placer.cc:162] value: (_Arg): /job:localhost/replica:0/task:0/device:CPU:0
2025-09-29 11:08:34.507539: I tensorflow/core/common_runtime/placer.cc:162] Fill: (Fill): /job:localhost/replica:0/task:0/device:CPU:0
2025-09-29 11:08:34.507545: I tensorflow/core/common_runtime/placer.cc:162] output_RetVal: (_Retval): /job:localhost/replica:0/task:0/device:CPU:0


Executing op MatMul in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op MatMul in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op Fill in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op RandomStandardNormal in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op Mul in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op AddV2 in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op RandomStandardNormal in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op Mul in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op AddV2 in device /job:localhost/replica:0/task:0/

...

2025-09-29 11:08:35.375382: I tensorflow/core/common_runtime/placer.cc:162] input: (_Arg): /job:localhost/replica:0/task:0/device:GPU:0
2025-09-29 11:08:35.375391: I tensorflow/core/common_runtime/placer.cc:162] _EagerConst: (_EagerConst): /job:localhost/replica:0/task:0/device:GPU:0
2025-09-29 11:08:35.375394: I tensorflow/core/common_runtime/placer.cc:162] output_RetVal: (_Retval): /job:localhost/replica:0/task:0/device:GPU:0
2025-09-29 11:08:35.465916: I tensorflow/core/common_runtime/placer.cc:162] x: (_DeviceArg): /job:localhost/replica:0/task:0/device:GPU:0
2025-09-29 11:08:35.465925: I tensorflow/core/common_runtime/placer.cc:162] Cast: (Cast): /job:localhost/replica:0/task:0/device:GPU:0
2025-09-29 11:08:35.465927: I tensorflow/core/common_runtime/placer.cc:162] y_RetVal: (_Retval): /job:localhost/replica:0/task:0/device:GPU:0


Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op FloorMod in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op Cast in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op StatelessRandomGetKeyCounter in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op StatelessRandomUniformV2 in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op Sub in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op Mul in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op AddV2 in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op VarHandleOp in device /job:lo

2025-09-29 11:08:35.666371: I tensorflow/core/common_runtime/placer.cc:162] input: (_Arg): /job:localhost/replica:0/task:0/device:GPU:0
2025-09-29 11:08:35.666382: I tensorflow/core/common_runtime/placer.cc:162] _EagerConst: (_EagerConst): /job:localhost/replica:0/task:0/device:GPU:0
2025-09-29 11:08:35.666384: I tensorflow/core/common_runtime/placer.cc:162] output_RetVal: (_Retval): /job:localhost/replica:0/task:0/device:GPU:0
2025-09-29 11:08:35.667510: I tensorflow/core/common_runtime/placer.cc:162] x: (_Arg): /job:localhost/replica:0/task:0/device:CPU:0
2025-09-29 11:08:35.667514: I tensorflow/core/common_runtime/placer.cc:162] y: (_Arg): /job:localhost/replica:0/task:0/device:CPU:0
2025-09-29 11:08:35.667517: I tensorflow/core/common_runtime/placer.cc:162] FloorMod: (FloorMod): /job:localhost/replica:0/task:0/device:CPU:0
2025-09-29 11:08:35.667519: I tensorflow/core/common_runtime/placer.cc:162] z_RetVal: (_Retval): /job:localhost/replica:0/task:0/device:CPU:0
2025-09-29 11:08:35

Executing op ReadVariableOp in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op BiasAdd in device /job:localhost/replica:0/task:0/device:GPU:0
TF forward pass done.


2025-09-29 11:08:35.884696: I tensorflow/core/common_runtime/placer.cc:162] resource: (_Arg): /job:localhost/replica:0/task:0/device:GPU:0
2025-09-29 11:08:35.884704: I tensorflow/core/common_runtime/placer.cc:162] ReadVariableOp: (ReadVariableOp): /job:localhost/replica:0/task:0/device:GPU:0
2025-09-29 11:08:35.884706: I tensorflow/core/common_runtime/placer.cc:162] value_RetVal: (_Retval): /job:localhost/replica:0/task:0/device:GPU:0


In [4]:
import tensorflow as tf, time

def bench_tf(device, n=4096, iters=3):
    with tf.device(device):
        a = tf.random.normal([n, n])
        b = tf.random.normal([n, n])
        c = tf.linalg.matmul(a, b)   # warmup
        _ = c.numpy()                # <-- forces sync
        t0 = time.time()
        for _ in range(iters):
            c = tf.linalg.matmul(a, b)
            _ = c.numpy()            # <-- forces sync each iter
    return time.time() - t0

cpu_t = bench_tf("/CPU:0")
gpu_t = bench_tf("/GPU:0")
print(f"TF CPU time: {cpu_t:.3f}s")
print(f"TF GPU time: {gpu_t:.3f}s")


Executing op RandomStandardNormal in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op Mul in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op AddV2 in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op RandomStandardNormal in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op Mul in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op AddV2 in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op MatMul in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op MatMul in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op MatMul in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op MatMul in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op RandomStandardNormal in device /job:localhost/replica:0/task:0/device:GPU:0
Executing op Mul in device /job:localhost/replica:0/task:0/de

In [5]:
#qucik pytorch mps test
import torch, time
print("Torch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

cpu = torch.device("cpu")
mps = torch.device("mps") if torch.backends.mps.is_available() else cpu

def bench(device, n=4096, iters=3):
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    c = a @ b  # warmup
    if device.type == "mps":
        torch.mps.synchronize()
    t0 = time.time()
    for _ in range(iters):
        c = a @ b
    if device.type == "mps":
        torch.mps.synchronize()
    return time.time() - t0

print("CPU:", bench(cpu))
if mps.type == "mps":
    print("MPS:", bench(mps))


Torch: 2.8.0
MPS available: True
CPU: 0.26898193359375
MPS: 0.16045212745666504
